# Phase 2 — Split and leakage audit

Run `python scripts/phase_02_acceptance.py` first. This notebook reads verified aggregate evidence; it does not train or open test predictions. Class counts on held-out data are permitted for partition integrity only.

Primary: nearest 70/15/15 Time-group boundaries. Reference: exact-record-group stratification, approximately 60/20/20 rows. Duplicates remain intact. SHA-256 uses canonical numeric source records including Time and Class. Source PCA provenance cannot be independently audited.

In [ ]:
import json
import pandas as pd
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from IPython import get_ipython
from matplotlib_inline.backend_inline import configure_inline_support
configure_inline_support(get_ipython(), 'module://matplotlib_inline.backend_inline')
from fraud_detection.paths import find_project_root
from fraud_detection.pipelines.train import load_verified_results

ROOT = find_project_root()
OUTPUT = ROOT / 'artifacts/phase_02'
result = load_verified_results(ROOT)
print('Source kind:', result['source_kind'])
print('Config SHA:', result['config_sha256'])
print('Test opened:', result['test_opened'])


In [ ]:
primary = json.loads((OUTPUT / 'split_manifest.json').read_text())
reference = json.loads((OUTPUT / 'reference_split_manifest.json').read_text())
print('Primary split SHA:', primary['split_sha256'])
print('Source SHA:', primary['dataset']['original_csv_sha256'])
for name, manifest in [('Primary', primary), ('Independent reference', reference)]:
    print(name, manifest['audit']['strategy'])
    display(pd.DataFrame(manifest['audit']['partitions']).T)
    print('Exact-record overlap:', manifest['audit']['exact_record_overlap'])
    print('Feature-conflict groups:', manifest['audit']['conflicting_feature_groups'])
summary = pd.DataFrame(primary['audit']['partitions']).T
summary['rows'].plot.bar(title='Original partition sizes', ylabel='Transactions')
plt.tight_layout()
plt.show()


## Review

Split indexes are hashed, not published as transaction rows. Time intervals are strictly separated. Every partition must contain both classes. Inspect drift in class prevalence without optimizing against held-out performance. Raw rows, row IDs and test scores are absent from this notebook.